## Script for histograms creation

In [35]:
%reload_ext autoreload
%autoreload 2

import glob
import json
import time
import os
import hashlib


import ROOT
import utils.physics_utils as PU
from utils.general_utils import get_valid_files_fast, log_analysis_run, build_cut_expression
from utils.hist_utils import (
    split_variable_group,
    fill_1d_histograms,
    fill_2d_histograms,
    draw_and_save_1d_histograms,
    draw_and_save_2d_histograms,
)

ROOT.EnableImplicitMT(90)
ROOT.gStyle.SetPalette(ROOT.kRainbow)

ROOT.gSystem.Load("/workspace/KLOE/Include/Codes/libLibRec.so")
ROOT.gSystem.Load("/workspace/KLOE/Subanalysis/CPFit/libCPFit.so")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/Codes/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Include/FortranAnalysis/inc")
ROOT.gInterpreter.AddIncludePath("/workspace/KLOE/src/Subanalysis/CPFit/inc")

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/FortranAnalysis/inc/fort_common.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/const.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/HypothesisCodes.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/SelectionCodes.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/charged_mom.h"')
ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Include/Codes/inc/kloe_class.h"')

ROOT.gInterpreter.Declare('#include "/workspace/KLOE/src/Subanalysis/CPFit/inc/cpfit.hpp"')

ROOT.KLOE.setGlobalStyle()

channName = {int(k): str(v) for k, v in ROOT.KLOE.channName}
channTitle = {str(k): str(v) for k, v in ROOT.KLOE.channTitle}
channColor = {str(k): int(v) for k, v in ROOT.KLOE.channColor}

# Enrichment for control samples
channTitle["has000"] = "Event includes K_{L} #rightarrow 3#pi^{0}"
channTitle["no000"] = "Event does not include K_{L} #rightarrow 3#pi^{0}"

channColor["has000"] = ROOT.kRed
channColor["no000"] = ROOT.kBlue

print(channName)

{0: 'Data', 1: 'Signal', 2: 'Regeneration', 3: 'Omega', 4: '3pi0', 5: 'Semileptonic', 6: 'Other', 7: 'pi+pi-pi+pi-', 8: 'MC sum'}


In [36]:
base_path = "/workspace/root_notebooks/results"

################ SETTINGS ##################################
analysis_dir = "ks_pm_control_sample"
img_dir = "img"
date = "2026-10-25"
root_file_name = "tag_000_1D_variables.root"
eff_graph_file_name = "efficiency_purity_graphs.root"
flags = ["has000", "no000", "Data", "MC sum"]
mode = "full"
nocut_hash = "nocuts"
log_flag = False
############################################################

hist_file_path = f"{base_path}/{analysis_dir}/{date}/{root_file_name}"
eff_graph_file_path = f"{base_path}/{analysis_dir}/{date}/{eff_graph_file_name}"

#################### ANALYSIS REGISTRY #####################
import pandas as pd
analysis_registry = f"{base_path}/{analysis_dir}/{date}/analysis_registry.csv"
df = pd.read_csv(analysis_registry)

hash_available = df["hash"]

## Making histograms

In [37]:
# Generate overlapped histograms for all channels
variables_1D = ["Knerec_5", 
                "KnerecSix_5", 
                "Kchrec_5",
                "Chi2SignalKinFit",
                "Chi2NeutralKinFit",
                "Chi2PMKinFit",
                "MassPi01Fit",
                "MassPi02Fit",
                "MassPi0NeutralFit1",
                "MassPi0NeutralFit2",
                "MassPi0PMFit1",
                "MassPi0PMFit2",
                "MassPi01",
                "MassPi02",
                "Delta_t_fit",
                "Delta_t_tri_fit",
                "Delta_t_rec",
                "nclu",
                "bestErrorSixGamma",
                "neu_clu_num"
                ]
variables_1D_titles = ["m^{inv}_{4#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{6#gamma} - m_{K^{0}} [MeV/c^{2}]", 
                       "m^{inv}_{+-} - m_{K^{0}} [MeV/c^{2}]",
                       "#chi^{2}_{+-00} [-]",
                       "#chi^{2}_{00} [-]",
                       "#chi^{2}_{+-} [-]",
                       "m^{inv,fit,+-00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,00}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,00}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,fit,+-}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,rec}_{#gamma#gamma,1} - m_{#pi^{0}} [MeV/c^{2}]",
                       "m^{inv,rec}_{#gamma#gamma,2} - m_{#pi^{0}} [MeV/c^{2}]",
                       "#Deltat_{fit} [#tau_{S}]",
                       "#Deltat_{tri} [#tau_{S}]",
                       "#Deltat_{rec} [#tau_{S}]",
                       "Total number of clusters",
                       "#Delta_{6} [cm^{2}]",
                       "Number of neutral clusters"
                       ]

unit_1D = ["MeV/c^{2}", 
            "MeV/c^{2}", 
            "MeV/c^{2}",
            "",
            "",
            "",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "MeV/c^{2}",
            "#tau_{S}",
            "#tau_{S}",
            "#tau_{S}",
            "",
            "cm^{2}",
            ""
            ]

range_X_1D = [(-400, 200), 
              (-400, 200), 
              (-200, 200),
              (0, 500),
              (0, 500),
              (0, 100),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-400, 400),
              (-300, 300),
              (-300, 300),
              (-300, 300),
              (0, 12),
              (0, 10000),
              (0, 12)
              ]

TRUTH_ONLY_VARS_1D = {}
TRUTH_MCTRUTH_ID_1D = 1

In [38]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)
ROOT.TH1.AddDirectory(False)

#################### OBJECT TO FIND ################################
histograms = {}

for hash in hash_available:
    if hash not in [key.GetName() for key in hist_file.GetListOfKeys()]:
        continue
    
    histograms[hash] = {}

    for var in variables_1D:
      dir_obj = hist_file.Get(f"{hash}/{var}")
      if var not in histograms[hash]:
          histograms[hash][var] = {}

      for flag in flags:
        hist_name = f"{var}_{flag}_{mode}"
        histograms[hash][var][flag] = dir_obj.Get(hist_name).Clone()

hist_file.Close()


In [39]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

################## DRAW USING THSTACK ###########################
canvases = []
legends = []
stacks = []

for hash in histograms:    
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)
    for i, var in enumerate(histograms[hash]):
        path_var = f"{path_hash}/{var}"
        os.makedirs(path_var, exist_ok=True)
        
        legend = ROOT.TLegend(0.15, 0.8, 0.84875, 0.9)
        legend.SetTextFont(62)
        legend.SetTextSize(0.025)
        legend.SetMargin(0.1)
        legend.SetNColumns(2)
        
        # Dodajemy hash do nazwy, aby uniknąć konfliktów w pamięci ROOTa przy wielu folderach
        stack = ROOT.THStack(f"{var}_stack_{hash}", "")
        
        for flag in histograms[hash][var]:
            hist = histograms[hash][var][flag]
            if hist:
                hist.SetLineColor(channColor[flag])

                if log_flag:
                    hist.SetMinimum(0.1)
                
                if flag != "Data":
                    # Opcję rysowania przekazujemy bezpośrednio w metodzie Add dla THStack
                    stack.Add(hist, "hist")
                    legend.AddEntry(hist, channTitle[flag], "l")
                else:
                    hist.SetMarkerStyle(20)
                    hist.SetMarkerColor(channColor[flag])
                    stack.Add(hist, "pe1")
                    legend.AddEntry(hist, channTitle[flag], "pe1")
                            
        canvas = ROOT.TCanvas(f"{var}_canvas_{hash}", f"{var} Canvas", 600, 600)
        
        stack.Draw("NOSTACK")
        
        if stack.GetHistogram():
            resolution = stack.GetHistogram().GetXaxis().GetBinWidth(1)
            
            stack.GetHistogram().GetXaxis().SetTitle(variables_1D_titles[i])
            stack.GetHistogram().GetYaxis().SetTitle(f"Counts / {resolution:.2f} {unit_1D[i]}")

        ####### Set log y ########
        if log_flag:
          stack.SetMinimum(0.1)
          stack.SetMaximum(stack.GetMaximum() * 50)
        else:
          stack.SetMinimum(0)
                
        canvas.SetLogy(log_flag)
        ##########################
            
        legend.Draw()

        canvas.Draw()
        canvas.SaveAs(f"{path_var}/{var}_{mode}_logy_{log_flag}.pdf")
        canvas.Close()

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Knerec_5/Knerec_5_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/KnerecSix_5/KnerecSix_5_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Kchrec_5/Kchrec_5_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Chi2SignalKinFit/Chi2SignalKinFit_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Chi2NeutralKinFit/Chi2NeutralKinFit_full_logy_False.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/Chi2PMKinFit/

## Check of purity in the function of $\Delta t$

In [34]:
import os

img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)



delta_t_vars = ["Delta_t_fit",
                "Delta_t_tri_fit",
                "Delta_t_rec"]

delta_t_titles = ["#Deltat_{fit} [#tau_{S}]",
                  "#Deltat_{tri} [#tau_{S}]",
                  "#Deltat_{rec} [#tau_{S}]"]

signal = "has000"
sum = "MC sum"

delta_t_signal = {}
delta_t_sum = {}

for hash in histograms:
    delta_t_signal[hash] = []
    delta_t_sum[hash] = []
    for var in delta_t_vars:
        if var not in histograms[hash]:
            continue

        delta_t_signal[hash].append(histograms[hash][var][signal])
        delta_t_sum[hash].append(histograms[hash][var][sum])


######################### DIVIDE #############################
delta_t_divided = {}

for hash in delta_t_signal:
    delta_t_divided[hash] = []
    for signal_hist, sum_hist in zip(delta_t_signal[hash], delta_t_sum[hash]):
        hist_temp = signal_hist.Clone()
        hist_temp.Divide(sum_hist)
        delta_t_divided[hash].append(hist_temp)


######################## PLOT ###########################
canvases = []

ROOT.gStyle.SetOptStat(0)

for hash in delta_t_divided:
    path_hash = f"{img_dir_path}/{hash}"
    os.makedirs(path_hash, exist_ok=True)
    for hist, title in zip(delta_t_divided[hash], delta_t_titles):
        c = ROOT.TCanvas("")
        hist.SetMarkerStyle(ROOT.kFullCircle)
        hist.SetLineColor(ROOT.kBlack)
        hist.GetYaxis().SetRangeUser(0, 1.1)
        hist.GetYaxis().SetTitle("Purity")
        hist.Draw("PE1")
        c.Draw()
        c.SaveAs(f"{path_hash}/purity_{title.replace(' ', '_')}.pdf")
        canvases.append(c)

        

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/purity_#Deltat_{fit}_[#tau_{S}].pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/purity_#Deltat_{tri}_[#tau_{S}].pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/nocuts/purity_#Deltat_{rec}_[#tau_{S}].pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/519c0397/purity_#Deltat_{fit}_[#tau_{S}].pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/519c0397/purity_#Deltat_{tri}_[#tau_{S}].pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/519c0397/purity_#Deltat_{rec}_[#tau_{S}].pdf has been created
Info

## Making efficiency - purity plots

In [4]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(eff_graph_file_path)
#################### OBJECT TO FIND ################################
graphs = {}
var_dict = {}

for hash in hash_available:
    if hash not in hist_file.GetListOfKeys():
        continue

    graphs[hash] = {}
    obj = hist_file.Get(hash)

    if obj:
      var_list = [key.GetName() for key in obj.GetListOfKeys()]
      var_dict[hash] = var_list
      for var in var_list:
          graphs[hash][var] = {}
          dir_obj = obj.Get(var)
          for type in ["efficiency", "purity"]:
            graphs[hash][var][type] = {}
            for flag in flags:
                graph_name = f"{type}_{flag}"
                graphs[hash][var][type][flag] = dir_obj.Get(graph_name)

hist_file.Close()

print("Graphs loaded:", graphs)

Graphs loaded: {'519c0397': {'bestErrorSixGamma': {'efficiency': {'has000': <cppyy.gbl.TGraph object at 0x562255aad480>, 'no000': <cppyy.gbl.TGraph object at 0x562255c30130>, 'Data': <cppyy.gbl.TGraph object at 0x562255c2e6d0>, 'MC sum': <cppyy.gbl.TGraph object at 0x562255bb3360>}, 'purity': {'has000': <cppyy.gbl.TGraph object at 0x562255b9bb40>, 'no000': <cppyy.gbl.TGraph object at 0x562255c4aaa0>, 'Data': <cppyy.gbl.TGraph object at 0x562255c2e0f0>, 'MC sum': <cppyy.gbl.TGraph object at 0x562255baa010>}}}, '66563976': {'Chi2NeutralKinFit': {'efficiency': {'has000': <cppyy.gbl.TGraph object at 0x562255c33970>, 'no000': <cppyy.gbl.TGraph object at 0x562255b7fbc0>, 'Data': <cppyy.gbl.TGraph object at 0x562255c45010>, 'MC sum': <cppyy.gbl.TGraph object at 0x562255c44cf0>}, 'purity': {'has000': <cppyy.gbl.TGraph object at 0x562255c44b60>, 'no000': <cppyy.gbl.TGraph object at 0x562255c34c00>, 'Data': <cppyy.gbl.TGraph object at 0x562255b486f0>, 'MC sum': <cppyy.gbl.TGraph object at 0x5622

In [5]:
img_dir_path = f"{base_path}/{analysis_dir}/{date}/{img_dir}"
os.makedirs(img_dir_path, exist_ok=True)

var_titles_graph = dict(zip(variables_1D, variables_1D_titles))

canvases = []
multigraphs = []
legends = []

################## DRAW USING THSTACK ###########################
for hash in hash_available:
  if hash not in graphs:
      continue
  path_hash = f"{img_dir_path}/{hash}/optimization"
  os.makedirs(path_hash, exist_ok=True)

  for i, var in enumerate(graphs[hash]):

    legend = ROOT.TLegend(0.5, 0.7, 0.9, 0.9)
    legend.SetTextFont(62)
    legend.SetTextSize(0.025)
    legend.SetMargin(0.1)

    mg = ROOT.TMultiGraph(f"mg_{hash}_{var}", f";{var_titles_graph[var]};Value")

    # Dodano opcję "l" (Line), żeby TMultiGraph wiedział, jak narysować TGraph
    mg.Add(graphs[hash][var]["efficiency"]["has000"], "l")
    mg.Add(graphs[hash][var]["purity"]["has000"], "l")
    mg.Add(graphs[hash][var]["efficiency"]["no000"], "l")

    legend.AddEntry(graphs[hash][var]["efficiency"]["has000"], "Efficiency for K_{L} #rightarrow 3#pi^{0}", "l")
    legend.AddEntry(graphs[hash][var]["purity"]["has000"], "Purity for K_{L} #rightarrow 3#pi^{0}", "l")
    legend.AddEntry(graphs[hash][var]["efficiency"]["no000"], "Efficiency without K_{L} #rightarrow 3#pi^{0}", "l")

    c = ROOT.TCanvas(f"c_graph_{var}_{hash}")
    canvases.append(c)
    multigraphs.append(mg)
    legends.append(legend)
    mg.Draw("AL")
    legend.Draw()
    
    # Wymagane odświeżenie przed zapisem, żeby ROOT poprawnie wyrenderował osie i linie
    c.Modified()
    c.Update()

    c.Draw()
    c.SaveAs(f"{path_hash}/{var}.pdf")
    # c.Close()

Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/519c0397/optimization/bestErrorSixGamma.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/66563976/optimization/Chi2NeutralKinFit.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/66563976/optimization/Knerec_5.pdf has been created
Info in <TCanvas::Print>: pdf file /workspace/root_notebooks/results/ks_pm_control_sample/2026-10-25/img/66563976/optimization/Kchrec_5.pdf has been created


## Calculating events, efficiencies and purities

In [6]:
#################### OPEN FILE | HISTOGRAMS ########################
hist_file = ROOT.TFile.Open(hist_file_path)

#################### OBJECT TO FIND ################################
event_counts = {}
cuts_list = {}
efficiencies = {}
purities = {}

#################### EXTRACT EVENT COUNTS #######################
for hash in hash_available:
    obj = hist_file.Get(f"{hash}/ChannelIntegrals")
    obj_cut_list = hist_file.Get(f"{hash}/CutList")

    if obj:
      chann_integrals_str = obj.GetTitle()
      event_counts[hash] = json.loads(chann_integrals_str)

    if obj_cut_list:
      raw_title = obj_cut_list.GetTitle()
      cleaned_str = raw_title.strip("[] ").replace('"', '')
      cuts_list[hash] = [c.strip() for c in cleaned_str.split(',') if c.strip()]

hist_file.Close()
#################################################################

#################### CALCULATE EFFICIENCIES AND PURITIES #######################
for hash in hash_available:
  if hash not in event_counts:
     continue

  efficiencies[hash] = {}
  purities[hash] = {}

  mc_sum = int(event_counts[hash].get("MC sum", 0))

  for flag in flags:
      total_events = int(event_counts.get(nocut_hash, None).get(flag, 0))
      passed_events = int(event_counts[hash].get(flag, 0))
      efficiencies[hash][flag] = passed_events / total_events if total_events > 0 else 0
      purities[hash][flag] = passed_events / mc_sum if mc_sum > 0 else 0
###############################################################################

################## PRINT RESULTS ##################
for hash in hash_available:
    if hash not in efficiencies:
        continue
    print(f"Results for cut list: {cuts_list.get(hash, [])}")
    for flag in flags:
        event_count = int(event_counts.get(hash, {}).get(flag, 0))
        mc_sum = int(event_counts.get(hash, {}).get("MC sum", 0))
        eff = efficiencies[hash].get(flag, 0)
        pur = purities[hash].get(flag, 0)
        print(f"  Flag: {flag}, Event Count: {event_count / 10**6:.2f} x 10^6, Efficiency: {eff * 100:.2f}%, Purity: {pur * 100:.2f}%, (MC - Data) / Data: {100*(mc_sum - event_count) / event_count if event_count > 0 else 0:.2f}%")
    print()




Results for cut list: []
  Flag: has000, Event Count: 91.00 x 10^6, Efficiency: 100.00%, Purity: 37.92%, (MC - Data) / Data: 163.72%
  Flag: no000, Event Count: 148.98 x 10^6, Efficiency: 100.00%, Purity: 62.08%, (MC - Data) / Data: 61.08%
  Flag: Data, Event Count: 230.93 x 10^6, Efficiency: 100.00%, Purity: 96.23%, (MC - Data) / Data: 3.92%
  Flag: MC sum, Event Count: 239.98 x 10^6, Efficiency: 100.00%, Purity: 100.00%, (MC - Data) / Data: 0.00%

Results for cut list: ['neu_clu_num >= 6']
  Flag: has000, Event Count: 83.88 x 10^6, Efficiency: 92.18%, Purity: 46.34%, (MC - Data) / Data: 115.80%
  Flag: no000, Event Count: 97.13 x 10^6, Efficiency: 65.20%, Purity: 53.66%, (MC - Data) / Data: 86.36%
  Flag: Data, Event Count: 170.70 x 10^6, Efficiency: 73.92%, Purity: 94.31%, (MC - Data) / Data: 6.04%
  Flag: MC sum, Event Count: 181.01 x 10^6, Efficiency: 75.43%, Purity: 100.00%, (MC - Data) / Data: 0.00%

Results for cut list: ['neu_clu_num >= 6', 'bestErrorSixGamma < 2000.0']
  Flag